# Elastic Network Modes with ElasNetMT

Elastic network models (ENMs) approximate protein flexibility as a network of beads
connected by harmonic springs.  The **elasnetmt** addon integrates ElasNetMT with
MolSysViewer to let you:

- build a GNM or ANM model from Cα atoms,
- visualise the contact network as link overlays,
- render displacement vectors for any normal mode,
- navigate modes interactively.

**Requirements:** `elasnetmt`, `molsysviewer`, `molsysviewer-elasnetmt`.

In [1]:
import molsysviewer as msv
from molsysviewer_elasnetmt import (
    addon as elasnetmt_addon,
    lifecycle,
    on_enable,
    on_context_action,
)
from molsysviewer_elasnetmt.runtime import ensure_runtime

## Register the addon

In [2]:
msv.addons.register(elasnetmt_addon, lifecycle=lifecycle)

AddonSpec(name='elasnetmt', package='elasnetmt', version=None, description='Elastic network analysis workspace for MolSysViewer.', workspaces=(AddonWorkspaceSpec(id='elasnetmt', title='Elastic Networks', entry_panel='modes', description='Workspace for GNM/ANM models and overlays.', order=40, meta={}),), panels=(AddonPanelSpec(id='model', title='Model', entry='molsysviewer_elasnetmt.panels.model', widget_class='molsysviewer_elasnetmt.panels.model.ElasNetMTModelPanel', description='Model parameters and node selection.', order=10, target='panel_mode', meta={}), AddonPanelSpec(id='modes', title='Modes', entry='molsysviewer_elasnetmt.panels.modes', widget_class='molsysviewer_elasnetmt.panels.modes.ElasNetMTModesPanel', description='Normal mode selection and activation.', order=20, target='panel_mode', meta={}), AddonPanelSpec(id='figures', title='Figures', entry='molsysviewer_elasnetmt.panels.figures', widget_class='molsysviewer_elasnetmt.panels.figures.ElasNetMTFiguresPanel', description='

## Create the view and enable the addon

We use triosephosphate isomerase (TIM, PDB 1TCD) as a compact, well-studied example.
The addon is enabled immediately after loading so that the runtime is attached to the view.

In [3]:
view = msv.MolSysView()
view.load("pdb_id:1tcd")
on_enable(view)
view.show()

## Build the elastic network model

The **Model** panel computes the GNM or ANM Kirchhoff/Hessian matrix and
performs the normal-mode decomposition.  `model_kind` can be `"gnm"` (Gaussian
Network Model) or `"anm"` (Anisotropic Network Model).

The cutoff distance (in ångströms) controls which Cα–Cα pairs are considered
bonded in the network.

In [4]:
runtime = ensure_runtime(view)
runtime.model_kind = "anm"
runtime.cutoff = "12 angstroms"

model_panel = view.addons.resolve_panel_widget("elasnetmt", "model")
model_panel.handle_action(view, "compute", {})

## Overlay the contact network

The contact network shows which Cα pairs are coupled by a spring.
It is rendered as link shapes coloured by spring strength.

In [5]:
on_context_action(
    view,
    "show-contact-network",
    {"addon": "elasnetmt", "addon_action_id": "show-contact-network"},
)

## Visualise displacement vectors for a normal mode

Each normal mode is a collective motion of the entire protein.  Mode 0 is the
slowest (largest-scale) motion.  The **Modes** panel renders the eigenvector as
displacement arrows on the Cα atoms.

In [6]:
modes_panel = view.addons.resolve_panel_widget("elasnetmt", "modes")

# Select mode 0 (slowest collective motion)
modes_panel.handle_action(view, "set_mode_index", {"mode_index": 0})

# Render the displacement vectors
modes_panel.handle_action(view, "show_mode_vectors", {})

## Navigate between modes

Modes are sorted by increasing eigenvalue (increasing frequency).  Incrementing the
index switches from the global breathing motion (mode 0) to progressively more
localised, higher-frequency fluctuations.

In [7]:
for mode_idx in [1, 2, 3]:
    modes_panel.handle_action(view, "set_mode_index", {"mode_index": mode_idx})
    modes_panel.handle_action(view, "show_mode_vectors", {})
    print(f"Showing mode {mode_idx}")

Showing mode 1
Showing mode 2
Showing mode 3


## Export a static snapshot

In [8]:
view.export.html("1tcd_anm_mode0.html", title="TIM — ANM mode 0")

## Next steps

- Switch `model_kind` to `"gnm"` to compare GNM vs ANM fluctuation profiles.
- Try the **Figures** panel to export mode-shape plots as publication-quality images.
- Overlay anisotropy ellipsoids via `on_context_action(view, "show-anisotropy-ellipsoids", ...)`.